# Main79 — Self-Contained Reproduction

**Purpose:** reproduce Main79 without depending on Manan's local `main66.py` import structure.

This notebook:
- reconstructs the leakage-safe Main64/Main78 teacher;
- trains the Main79-style 3-seed BiGRU residual ensemble;
- evaluates the residual ensemble on the canonical 80/20 validation split;
- rebuilds the exact Main66 full-data test teacher;
- creates a self-contained Main79 test submission;
- saves validation and test diagnostic arrays for later stacking.

**Important:** this is a reproduction/diagnostic notebook. Do not submit anything to Kaggle automatically.


## 0. Upload files

Upload these files into Colab when prompted:

`train.json`, `test.json`, `main66.py`, `main64_oof_meta_features.npy`, `main64_val_meta_features.npy`, `main64_oof_svm.npy`, `main64_oof_nbsvm.npy`, `main64_oof_hgb.npy`, `main64_oof_local.npy`.

The helper below accepts filenames with suffixes such as `(1)` / `(2)`.


In [ ]:

from google.colab import files
import os, glob, shutil, pathlib, re

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

def resolve_uploaded(stem):
    candidates = []
    for p in glob.glob("/content/*"):
        name=os.path.basename(p)
        if name==stem or name.startswith(stem+"(") or name.startswith(stem+"."):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError(f"Could not find uploaded file starting with: {stem}")
    candidates.sort(key=lambda p: (0 if os.path.basename(p)==stem else 1, len(p)))
    return candidates[0]

PATHS = {
    "train": resolve_uploaded("train.json"),
    "test": resolve_uploaded("test.json"),
    "main66": resolve_uploaded("main66.py"),
    "oof_meta": resolve_uploaded("main64_oof_meta_features.npy"),
    "val_meta": resolve_uploaded("main64_val_meta_features.npy"),
    "oof_svm": resolve_uploaded("main64_oof_svm.npy"),
    "oof_nbsvm": resolve_uploaded("main64_oof_nbsvm.npy"),
    "oof_hgb": resolve_uploaded("main64_oof_hgb.npy"),
    "oof_local": resolve_uploaded("main64_oof_local.npy"),
}
for k,v in PATHS.items(): print(k, "->", v)


## 1. Load data and verify the Main64 artifacts

In [ ]:

import json, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

def load_jsonl(path,labelled=True):
    rows=[]
    with open(path,encoding="utf-8") as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    ids=[r["id"] for r in rows]
    texts=[r["text"] for r in rows]
    if labelled:
        y=np.asarray([0 if r["label"]=="A" else 1 for r in rows],dtype=np.int64)
        return ids,texts,y
    return ids,texts

train_ids,texts,labels=load_jsonl(PATHS["train"],True)
test_ids,test_texts=load_jsonl(PATHS["test"],False)

idx=np.arange(len(labels))
train_idx,val_idx=train_test_split(idx,test_size=.20,random_state=42,stratify=labels)

oof_meta=np.load(PATHS["oof_meta"]).astype(np.float32)
val_meta=np.load(PATHS["val_meta"]).astype(np.float32)

print("Train:",len(texts),"Test:",len(test_texts))
print("Canonical split:",len(train_idx),len(val_idx))
print("OOF meta:",oof_meta.shape,"VAL meta:",val_meta.shape)

assert len(texts)==10536 and len(test_texts)==3000
assert oof_meta.shape==(8428,13)
assert val_meta.shape==(2108,13)


## 2. Embed the exact Main66 implementation

In [ ]:
# The following cell is the exact uploaded main66.py source, executed in-memory so the notebook is self-contained.
MAIN66_SOURCE = '\nimport json\nimport joblib\nimport os\nimport time\nimport warnings\n\nimport numpy as np\nimport pandas as pd\n\nfrom scipy.sparse import hstack, csr_matrix\nfrom sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import LinearSVC\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.neighbors import NearestNeighbors\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import accuracy_score, confusion_matrix\n\nwarnings.filterwarnings("ignore")\n\n# ============================================================\n# MAIN66\n# Final submission pipeline\n#\n# Strategy:\n#   1. Reproduce Main64 validation pipeline.\n#   2. Apply calibrated probability threshold = 0.3745.\n#   3. If validation sanity check passes, refit base models\n#      on ALL 10,536 labelled documents.\n#   4. Predict test.json.\n#   5. Write submission.csv.\n#\n# NOTE:\n# The meta-model is intentionally the validated Main64\n# meta-model, rather than another expensive 5-fold OOF run.\n# ============================================================\n\nSEED = 42\n\nWORD_NGRAM = (1, 6)\nWORD_MIN_DF = 3\nTRANS_NGRAM = (1, 2)\nTRANS_MIN_DF = 2\n\nSVM_C = 10.0\nNBSVM_C = 10.0\nLOCAL_K = 20\nMETA_C = 0.1\n\nCALIBRATED_THRESHOLD = 0.3745\n\nTRAIN_FILE = "train.json"\nTEST_FILE = "test.json"\nSUBMISSION_FILE = "submission.csv"\n\nEXPECTED_VALIDATION = 0.930266\nVALIDATION_TOLERANCE = 0.015\n\n# ============================================================\n# DATA\n# ============================================================\n\ndef load_jsonl(path, labelled=True):\n    records = []\n\n    with open(path, "r") as f:\n        for line in f:\n            line = line.strip()\n            if line:\n                records.append(json.loads(line))\n\n    texts = [r["text"] for r in records]\n    ids = [r["id"] for r in records]\n\n    if labelled:\n        labels = np.array(\n            [0 if r["label"] == "A" else 1 for r in records],\n            dtype=np.int64\n        )\n        return ids, texts, labels\n\n    return ids, texts\n\n\ndef to_strings(texts):\n    return [" ".join(map(str, doc)) for doc in texts]\n\n\ndef make_transition_strings(texts):\n    output = []\n\n    for doc in texts:\n        if len(doc) < 2:\n            output.append("")\n            continue\n\n        transitions = [\n            f"{doc[i]}T{doc[i + 1]}"\n            for i in range(len(doc) - 1)\n        ]\n\n        output.append(" ".join(transitions))\n\n    return output\n\n\n# ============================================================\n# COMPACT FEATURES — EXACTLY MAIN64\n# ============================================================\n\ndef entropy_from_counts(counts):\n    if len(counts) == 0:\n        return 0.0\n\n    counts = np.asarray(counts, dtype=np.float64)\n    total = counts.sum()\n\n    if total <= 0:\n        return 0.0\n\n    p = counts / total\n    return float(-(p * np.log(p + 1e-12)).sum())\n\n\ndef sequence_features(doc):\n    x = np.asarray(doc, dtype=np.int64)\n    n = len(x)\n\n    if n == 0:\n        return np.zeros(62, dtype=np.float32)\n\n    unique, counts = np.unique(x, return_counts=True)\n    u = len(unique)\n\n    repetition_ratio = 1.0 - (u / n)\n    max_freq = counts.max()\n\n    repeated_occurrences = np.sum(counts[counts > 1] - 1)\n    repeated_types = np.sum(counts > 1)\n\n    entropy = entropy_from_counts(counts)\n\n    def ngram_stats(k):\n        if n < k:\n            return 0.0, 0.0\n\n        grams = set()\n        for i in range(n - k + 1):\n            grams.add(tuple(x[i:i + k]))\n\n        total = n - k + 1\n        return len(grams) / total, len(grams)\n\n    bdiv, buniq = ngram_stats(2)\n    tdiv, tuniq = ngram_stats(3)\n\n    quarter_values = []\n\n    for q in range(4):\n        start = (q * n) // 4\n        end = ((q + 1) * n) // 4\n        part = x[start:end]\n\n        if len(part) == 0:\n            quarter_values.extend([0.0] * 5)\n            continue\n\n        pu, pc = np.unique(part, return_counts=True)\n        plen = len(part)\n\n        quarter_values.extend([\n            plen / n,\n            len(pu) / plen,\n            entropy_from_counts(pc),\n            1.0 - len(pu) / plen,\n            pc.max() / plen\n        ])\n\n    k = min(10, n)\n\n    beginning = x[:k]\n    ending = x[-k:]\n\n    begin_unique = len(np.unique(beginning)) / k\n    end_unique = len(np.unique(ending)) / k\n\n    begin_entropy = entropy_from_counts(\n        np.unique(beginning, return_counts=True)[1]\n    )\n    end_entropy = entropy_from_counts(\n        np.unique(ending, return_counts=True)[1]\n    )\n\n    mid = n // 2\n    first = x[:mid]\n    second = x[mid:]\n\n    if len(first) > 0:\n        first_unique_ratio = len(np.unique(first)) / len(first)\n        first_entropy = entropy_from_counts(\n            np.unique(first, return_counts=True)[1]\n        )\n    else:\n        first_unique_ratio = 0.0\n        first_entropy = 0.0\n\n    if len(second) > 0:\n        second_unique_ratio = len(np.unique(second)) / len(second)\n        second_entropy = entropy_from_counts(\n            np.unique(second, return_counts=True)[1]\n        )\n    else:\n        second_unique_ratio = 0.0\n        second_entropy = 0.0\n\n    zero_count = np.sum(x == 0)\n    zero_ratio = zero_count / n\n\n    if n > 1:\n        same_adjacent = np.sum(x[1:] == x[:-1])\n        adjacent_repeat_ratio = same_adjacent / (n - 1)\n\n        transitions = np.stack([x[:-1], x[1:]], axis=1)\n        unique_transitions = len(\n            np.unique(transitions, axis=0)\n        )\n        transition_diversity = unique_transitions / (n - 1)\n    else:\n        adjacent_repeat_ratio = 0.0\n        transition_diversity = 0.0\n\n    max_freq_ratio = max_freq / n\n    repeated_type_ratio = repeated_types / max(u, 1)\n\n    features = [\n        np.log1p(n), n,\n        np.log1p(u), u,\n        u / n, repetition_ratio,\n        max_freq, max_freq_ratio,\n        repeated_occurrences, repeated_occurrences / n,\n        repeated_types, repeated_type_ratio,\n        entropy, entropy / np.log(max(u, 2)),\n        zero_count, zero_ratio,\n\n        bdiv, buniq,\n        tdiv, tuniq,\n        transition_diversity,\n        adjacent_repeat_ratio,\n\n        begin_unique, end_unique,\n        begin_entropy, end_entropy,\n\n        first_unique_ratio, second_unique_ratio,\n        first_entropy, second_entropy,\n        second_unique_ratio - first_unique_ratio,\n        second_entropy - first_entropy,\n\n        np.std(counts),\n        np.mean(counts),\n        np.median(counts),\n        np.max(counts) - np.median(counts),\n\n        *quarter_values\n    ]\n\n    return np.asarray(features, dtype=np.float32)\n\n\ndef build_compact_features(texts):\n    return np.vstack([sequence_features(doc) for doc in texts])\n\n\n# ============================================================\n# BASE MODELS — EXACTLY MAIN64 CONVENTIONS\n# ============================================================\n\ndef train_svm(X_train, y_train, X_query):\n    model = LinearSVC(\n        C=SVM_C,\n        class_weight="balanced",\n        tol=1e-2,\n        max_iter=500000\n    )\n    model.fit(X_train, y_train)\n\n    # Positive = B\n    return model.decision_function(X_query)\n\n\ndef train_nbsvm(train_strings, y_train, query_strings):\n    vectorizer = CountVectorizer(\n        ngram_range=(1, 3),\n        min_df=2,\n        binary=True\n    )\n\n    X_train = vectorizer.fit_transform(train_strings)\n    X_query = vectorizer.transform(query_strings)\n\n    A = X_train[y_train == 0]\n    B = X_train[y_train == 1]\n\n    alpha = 1.0\n\n    pA = np.asarray(A.sum(axis=0)).ravel() + alpha\n    pB = np.asarray(B.sum(axis=0)).ravel() + alpha\n\n    pA /= pA.sum()\n    pB /= pB.sum()\n\n    ratio = np.log(pA / pB)\n\n    X_train_nb = X_train.multiply(ratio)\n    X_query_nb = X_query.multiply(ratio)\n\n    model = LinearSVC(\n        C=NBSVM_C,\n        class_weight="balanced",\n        tol=1e-2,\n        max_iter=500000\n    )\n\n    model.fit(X_train_nb, y_train)\n\n    # Positive = B\n    return model.decision_function(X_query_nb)\n\n\ndef train_hgb(X_train, y_train, X_query):\n    model = HistGradientBoostingClassifier(\n        learning_rate=0.08,\n        max_iter=300,\n        max_leaf_nodes=31,\n        min_samples_leaf=10,\n        l2_regularization=1.0,\n        random_state=SEED\n    )\n\n    weights = np.where(\n        y_train == 0,\n        1.0,\n        np.sum(y_train == 0) / np.sum(y_train == 1)\n    )\n\n    model.fit(\n        X_train,\n        y_train,\n        sample_weight=weights\n    )\n\n    probability_B = model.predict_proba(X_query)[:, 1]\n\n    # Positive = B\n    return probability_B - 0.5\n\n\ndef local_geometry_features(\n    X_reference,\n    y_reference,\n    X_query,\n    k=20\n):\n    """\n    A-oriented geometry.\n    Positive = A, negative = B.\n    """\n\n    A_mask = y_reference == 0\n    B_mask = y_reference == 1\n\n    XA = X_reference[A_mask]\n    XB = X_reference[B_mask]\n\n    kA = min(k, XA.shape[0])\n    kB = min(k, XB.shape[0])\n\n    nnA = NearestNeighbors(\n        n_neighbors=kA,\n        metric="cosine",\n        algorithm="brute",\n        n_jobs=-1\n    )\n\n    nnB = NearestNeighbors(\n        n_neighbors=kB,\n        metric="cosine",\n        algorithm="brute",\n        n_jobs=-1\n    )\n\n    nnA.fit(XA)\n    nnB.fit(XB)\n\n    distA, _ = nnA.kneighbors(X_query)\n    distB, _ = nnB.kneighbors(X_query)\n\n    simA = 1.0 - distA\n    simB = 1.0 - distB\n\n    output = []\n\n    for i in range(X_query.shape[0]):\n        gap1 = distB[i, 0] - distA[i, 0]\n        similarity_gap1 = simA[i, 0] - simB[i, 0]\n\n        mean_similarity_gap = (\n            np.mean(simA[i]) - np.mean(simB[i])\n        )\n\n        weights_A = np.exp(5.0 * simA[i])\n        weights_B = np.exp(5.0 * simB[i])\n\n        vote_A = np.sum(weights_A)\n        vote_B = np.sum(weights_B)\n\n        weighted_vote = (\n            (vote_A - vote_B) /\n            (vote_A + vote_B + 1e-12)\n        )\n\n        density_gap = (\n            np.mean(simA[i]) - np.mean(simB[i])\n        )\n\n        multi = []\n\n        for kk in [1, 2, 3, 5, 10, 20]:\n            kkA = min(kk, kA)\n            kkB = min(kk, kB)\n\n            mean_A = np.mean(simA[i, :kkA])\n            mean_B = np.mean(simB[i, :kkB])\n\n            multi.append(mean_A - mean_B)\n\n        output.append([\n            gap1,\n            similarity_gap1,\n            mean_similarity_gap,\n            weighted_vote,\n            density_gap,\n            *multi\n        ])\n\n    return np.asarray(output, dtype=np.float64)\n\n\n# ============================================================\n# BUILD REPRESENTATIONS\n# ============================================================\n\ndef build_representation(texts, fit_data=None):\n    """\n    fit_data is unused here; kept only to make the workflow clear.\n    Vectorizers are always fitted explicitly where needed.\n    """\n    strings = to_strings(texts)\n    transitions = make_transition_strings(texts)\n\n    return strings, transitions\n\n\ndef fit_global_representation(train_texts):\n    train_strings = to_strings(train_texts)\n    train_transitions = make_transition_strings(train_texts)\n\n    tfidf = TfidfVectorizer(\n        ngram_range=WORD_NGRAM,\n        min_df=WORD_MIN_DF,\n        sublinear_tf=True\n    )\n\n    X_tfidf = tfidf.fit_transform(train_strings)\n\n    trans_vectorizer = TfidfVectorizer(\n        ngram_range=TRANS_NGRAM,\n        min_df=TRANS_MIN_DF,\n        sublinear_tf=True,\n        token_pattern=r"(?u)\\S+"\n    )\n\n    X_transition = trans_vectorizer.fit_transform(train_transitions)\n\n    compact = build_compact_features(train_texts)\n\n    scaler = StandardScaler()\n    compact_s = scaler.fit_transform(compact)\n\n    X_global = hstack([\n        X_tfidf,\n        csr_matrix(compact_s),\n        X_transition\n    ]).tocsr()\n\n    return (\n        train_strings,\n        train_transitions,\n        tfidf,\n        trans_vectorizer,\n        scaler,\n        X_tfidf,\n        compact,\n        X_global\n    )\n\n\ndef transform_global_representation(\n    texts,\n    tfidf,\n    trans_vectorizer,\n    compact_scaler\n):\n    strings = to_strings(texts)\n    transitions = make_transition_strings(texts)\n\n    X_tfidf = tfidf.transform(strings)\n    X_transition = trans_vectorizer.transform(transitions)\n\n    compact = build_compact_features(texts)\n    compact_s = compact_scaler.transform(compact)\n\n    X_global = hstack([\n        X_tfidf,\n        csr_matrix(compact_s),\n        X_transition\n    ]).tocsr()\n\n    return (\n        strings,\n        transitions,\n        X_tfidf,\n        compact,\n        X_global\n    )\n\n\n# ============================================================\n# META FEATURE CONSTRUCTION\n# ============================================================\n\ndef make_meta_features(\n    svm,\n    nbsvm,\n    hgb,\n    local,\n    scaler,\n    fit_scaler=False\n):\n    # local is A-positive, so convert to B-positive\n    local_b = -local\n\n    raw = np.column_stack([\n        svm,\n        nbsvm,\n        hgb,\n        local_b[:, 0],\n        local_b[:, 1],\n        local_b[:, 2],\n        local_b[:, 3],\n        local_b[:, 6],\n        local_b[:, 7],\n        local_b[:, 8],\n        local_b[:, 9],\n        local_b[:, 10]\n    ])\n\n    if fit_scaler:\n        transformed = scaler.fit_transform(raw)\n    else:\n        transformed = scaler.transform(raw)\n\n    uncertainty = np.exp(-np.abs(svm))\n    geometry_signal = local_b[:, 3]\n\n    interaction = (\n        geometry_signal * uncertainty\n    ).reshape(-1, 1)\n\n    return np.hstack([transformed, interaction])\n\n\n# ============================================================\n# MAIN\n# ============================================================\n\ndef main():\n    start = time.time()\n\n    print("=" * 70)\n    print("MAIN66")\n    print("Final calibrated submission pipeline")\n    print("=" * 70)\n\n    # --------------------------------------------------------\n    # Load labelled data\n    # --------------------------------------------------------\n\n    train_ids, texts, labels = load_jsonl(\n        TRAIN_FILE,\n        labelled=True\n    )\n\n    print("\\nDocuments:", len(texts))\n    print("Class A:", np.sum(labels == 0))\n    print("Class B:", np.sum(labels == 1))\n\n    # --------------------------------------------------------\n    # Recreate exact Main64 external validation split\n    # --------------------------------------------------------\n\n    indices = np.arange(len(texts))\n\n    train_idx, val_idx = train_test_split(\n        indices,\n        test_size=0.20,\n        random_state=SEED,\n        stratify=labels\n    )\n\n    train_texts = [texts[i] for i in train_idx]\n    val_texts = [texts[i] for i in val_idx]\n\n    y_train = labels[train_idx]\n    y_val = labels[val_idx]\n\n    print("\\nValidation sanity stage")\n    print("Training:", len(train_idx))\n    print("Validation:", len(val_idx))\n\n    # --------------------------------------------------------\n    # Fit representation on 8428 training documents\n    # --------------------------------------------------------\n\n    print("\\nFitting validation-stage representation...")\n\n    (\n        train_strings,\n        train_transitions,\n        tfidf,\n        trans_vectorizer,\n        compact_scaler,\n        X_train_tfidf,\n        X_train_compact,\n        X_train_global\n    ) = fit_global_representation(train_texts)\n\n    (\n        val_strings,\n        val_transitions,\n        X_val_tfidf,\n        X_val_compact,\n        X_val_global\n    ) = transform_global_representation(\n        val_texts,\n        tfidf,\n        trans_vectorizer,\n        compact_scaler\n    )\n\n    print("Global TF-IDF:", X_train_tfidf.shape)\n    print("Transition TF-IDF:", trans_vectorizer.transform(train_transitions).shape)\n    print("Compact:", X_train_compact.shape)\n    print("Combined:", X_train_global.shape)\n\n    # --------------------------------------------------------\n    # Fit validation base models\n    # --------------------------------------------------------\n\n    print("\\nTraining validation SVM...")\n    val_svm = train_svm(\n        X_train_global,\n        y_train,\n        X_val_global\n    )\n\n    print("Training validation NBSVM...")\n    val_nbsvm = train_nbsvm(\n        train_strings,\n        y_train,\n        val_strings\n    )\n\n    print("Training validation HGB...")\n    val_hgb = train_hgb(\n        X_train_compact,\n        y_train,\n        X_val_compact\n    )\n\n    print("Computing validation local geometry...")\n    val_local = local_geometry_features(\n        X_train_tfidf,\n        y_train,\n        X_val_tfidf,\n        k=LOCAL_K\n    )\n\n    # --------------------------------------------------------\n    # Load Main64 OOF features to recreate its meta-model\n    # --------------------------------------------------------\n\n    print("\\nLoading Main64 OOF meta-features...")\n\n    oof_meta_path = "main64_oof_meta_features.npy"\n\n    if not os.path.exists(oof_meta_path):\n        raise FileNotFoundError(\n            f"Missing {oof_meta_path}. "\n            "Keep the Main64 result files in the project folder."\n        )\n\n    oof_meta = np.load(oof_meta_path)\n\n    if oof_meta.shape[1] != 13:\n        raise ValueError(\n            f"Expected 13 Main64 meta features, got {oof_meta.shape}"\n        )\n\n    print("OOF meta shape:", oof_meta.shape)\n\n    # Recreate Main64 meta model.\n    # Main64 meta features were already standardized and include\n    # the final interaction column.\n    meta_model = LogisticRegression(\n        C=META_C,\n        class_weight="balanced",\n        max_iter=5000,\n        random_state=SEED\n    )\n\n    meta_model.fit(oof_meta, y_train)\n\n    # Main64\'s scaler is not saved separately, so reconstruct it\n    # from OOF meta features after removing the interaction column.\n    #\n    # The OOF file contains:\n    #   12 standardized base features + interaction.\n    #\n    # We therefore standardize the new validation base features\n    # using the distribution implied by Main64\'s OOF feature file.\n    #\n    # The first 12 OOF columns are zero mean/unit variance, so this\n    # is equivalent to applying their OOF means/stds (0 and 1).\n    #\n    # For exact transfer, use the saved Main64 validation meta\n    # distribution as a diagnostic and keep the same score units.\n    #\n    # In practice Main64\'s saved OOF meta is already standardized,\n    # so validation raw features are standardized independently\n    # below using the OOF base score arrays.\n\n    oof_svm = np.load("main64_oof_svm.npy")\n    oof_nbsvm = np.load("main64_oof_nbsvm.npy")\n    oof_hgb = np.load("main64_oof_hgb.npy")\n    oof_local = np.load("main64_oof_local.npy")\n\n    # Reconstruct the exact Main64 meta scaler from raw OOF scores.\n    oof_local_b = -oof_local\n\n    oof_raw = np.column_stack([\n        oof_svm,\n        oof_nbsvm,\n        oof_hgb,\n        oof_local_b[:, 0],\n        oof_local_b[:, 1],\n        oof_local_b[:, 2],\n        oof_local_b[:, 3],\n        oof_local_b[:, 6],\n        oof_local_b[:, 7],\n        oof_local_b[:, 8],\n        oof_local_b[:, 9],\n        oof_local_b[:, 10]\n    ])\n\n    exact_meta_scaler = StandardScaler()\n    exact_meta_scaler.fit(oof_raw)\n\n    # Sanity: this should recreate Main64\'s first 12 columns.\n    reconstructed_oof_base = exact_meta_scaler.transform(oof_raw)\n\n    max_diff = np.max(\n        np.abs(reconstructed_oof_base - oof_meta[:, :12])\n    )\n\n    print("Meta-feature reconstruction max difference:", max_diff)\n\n    if max_diff > 1e-5:\n        raise RuntimeError(\n            "Main64 meta-feature reconstruction failed. "\n            "Stopping before submission."\n        )\n\n    # Build validation meta features\n    val_meta = make_meta_features(\n        val_svm,\n        val_nbsvm,\n        val_hgb,\n        val_local,\n        exact_meta_scaler\n    )\n\n    val_probability_B = meta_model.predict_proba(\n        val_meta\n    )[:, 1]\n\n    # --------------------------------------------------------\n    # VALIDATION CHECK\n    # --------------------------------------------------------\n\n    print("\\n" + "=" * 70)\n    print("FINAL VALIDATION SANITY CHECK")\n    print("=" * 70)\n\n    default_pred = (\n        val_probability_B >= 0.5\n    ).astype(int)\n\n    calibrated_pred = (\n        val_probability_B >= CALIBRATED_THRESHOLD\n    ).astype(int)\n\n    default_acc = accuracy_score(\n        y_val,\n        default_pred\n    )\n\n    calibrated_acc = accuracy_score(\n        y_val,\n        calibrated_pred\n    )\n\n    print(\n        f"Main64 threshold 0.5000 : "\n        f"{default_acc * 100:.4f}%"\n    )\n\n    print(\n        f"Calibrated threshold "\n        f"{CALIBRATED_THRESHOLD:.4f} : "\n        f"{calibrated_acc * 100:.4f}%"\n    )\n\n    print("\\nCalibrated confusion:")\n    print(confusion_matrix(y_val, calibrated_pred))\n\n    print("\\nErrors:", np.sum(calibrated_pred != y_val))\n\n    # Orientation sanity\n    print("\\nPrediction counts:")\n    print("A:", np.sum(calibrated_pred == 0))\n    print("B:", np.sum(calibrated_pred == 1))\n\n    if calibrated_acc < EXPECTED_VALIDATION - VALIDATION_TOLERANCE:\n        raise RuntimeError(\n            "\\nVALIDATION SANITY CHECK FAILED.\\n"\n            f"Expected approximately {EXPECTED_VALIDATION * 100:.2f}% "\n            f"but obtained {calibrated_acc * 100:.2f}%.\\n"\n            "No submission was generated."\n        )\n\n    print("\\nVALIDATION SANITY CHECK: PASS")\n\n    # --------------------------------------------------------\n    # LOAD TEST DATA\n    # --------------------------------------------------------\n\n    print("\\n" + "=" * 70)\n    print("LOADING TEST DATA")\n    print("=" * 70)\n\n    test_ids, test_texts = load_jsonl(\n        TEST_FILE,\n        labelled=False\n    )\n\n    print("Test documents:", len(test_texts))\n\n    if len(test_texts) != 3000:\n        print(\n            "WARNING: expected 3000 test documents, "\n            f"found {len(test_texts)}"\n        )\n\n    # --------------------------------------------------------\n    # FIT REPRESENTATION ON ALL LABELLED DATA\n    # --------------------------------------------------------\n\n    print("\\n" + "=" * 70)\n    print("RETRAINING BASE MODELS ON ALL LABELLED DATA")\n    print("=" * 70)\n\n    (\n        all_strings,\n        all_transitions,\n        full_tfidf,\n        full_trans_vectorizer,\n        full_compact_scaler,\n        X_all_tfidf,\n        X_all_compact,\n        X_all_global\n    ) = fit_global_representation(texts)\n\n    print("Full TF-IDF:", X_all_tfidf.shape)\n    print("Full compact:", X_all_compact.shape)\n    print("Full combined:", X_all_global.shape)\n\n    (\n        test_strings,\n        test_transitions,\n        X_test_tfidf,\n        X_test_compact,\n        X_test_global\n    ) = transform_global_representation(\n        test_texts,\n        full_tfidf,\n        full_trans_vectorizer,\n        full_compact_scaler\n    )\n\n    # --------------------------------------------------------\n    # FINAL BASE MODELS\n    # --------------------------------------------------------\n\n    print("\\nFinal SVM...")\n    test_svm = train_svm(\n        X_all_global,\n        labels,\n        X_test_global\n    )\n\n    print("Final NBSVM...")\n    test_nbsvm = train_nbsvm(\n        all_strings,\n        labels,\n        test_strings\n    )\n\n    print("Final HGB...")\n    test_hgb = train_hgb(\n        X_all_compact,\n        labels,\n        X_test_compact\n    )\n\n    print("Final local geometry...")\n    test_local = local_geometry_features(\n        X_all_tfidf,\n        labels,\n        X_test_tfidf,\n        k=LOCAL_K\n    )\n\n    # --------------------------------------------------------\n    # FINAL META PREDICTION\n    # --------------------------------------------------------\n\n    print("\\nBuilding final meta features...")\n\n    test_meta = make_meta_features(\n        test_svm,\n        test_nbsvm,\n        test_hgb,\n        test_local,\n        exact_meta_scaler\n    )\n\n    test_probability_B = meta_model.predict_proba(\n        test_meta\n    )[:, 1]\n\n    test_prediction = (\n        test_probability_B >= CALIBRATED_THRESHOLD\n    ).astype(int)\n\n    test_labels = np.where(\n        test_prediction == 0,\n        "A",\n        "B"\n    )\n\n    # --------------------------------------------------------\n    # SANITY CHECKS\n    # --------------------------------------------------------\n\n    print("\\n" + "=" * 70)\n    print("TEST SANITY CHECKS")\n    print("=" * 70)\n\n    assert len(test_ids) == len(test_prediction)\n    assert len(test_ids) == len(test_labels)\n    assert len(test_ids) == len(test_probability_B)\n\n    assert len(set(test_ids)) == len(test_ids)\n\n    assert np.all(\n        np.isin(test_labels, ["A", "B"])\n    )\n\n    assert np.all(\n        np.isfinite(test_probability_B)\n    )\n\n    print("Prediction count:", len(test_prediction), "PASS")\n    print("Unique IDs:", len(set(test_ids)), "PASS")\n    print("Labels only A/B: PASS")\n    print("No NaN/Inf scores: PASS")\n\n    print("\\nTest predictions:")\n    print("A:", np.sum(test_labels == "A"))\n    print("B:", np.sum(test_labels == "B"))\n\n    # --------------------------------------------------------\n    # SUBMISSION CSV\n    # --------------------------------------------------------\n\n    print("\\n" + "=" * 70)\n    print("CREATING SUBMISSION")\n    print("=" * 70)\n\n    # Competition-specific submission schemas vary.\n    # The assignment\'s expected schema is ID + label.\n    submission = pd.DataFrame({\n        "id": test_ids,\n        "label": test_labels\n    })\n\n    submission.to_csv(\n        SUBMISSION_FILE,\n        index=False\n    )\n\n    # Re-read to verify what was actually written.\n    check = pd.read_csv(SUBMISSION_FILE)\n\n    assert len(check) == len(test_ids)\n    assert list(check.columns) == ["id", "label"]\n    assert check["id"].is_unique\n    assert set(check["label"].unique()).issubset({"A", "B"})\n\n    print("\\nSUBMISSION CREATED SUCCESSFULLY")\n    print("File:", os.path.abspath(SUBMISSION_FILE))\n    print("Rows:", len(check))\n    print("Columns:", list(check.columns))\n\n\nif __name__ == "__main__":\n    main()\n    print("A:", np.sum(check["label"] == "A"))\n    print("B:", np.sum(check["label"] == "B"))\n\n    print("\\n" + "=" * 70)\n    print("DONE")\n    print("=" * 70)\n\n    print(\n        f"\\nTotal time: {time.time() - start:.2f} seconds"\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
exec(MAIN66_SOURCE, globals())

## 3. Main79 teacher + residual implementation

In [ ]:

# =========================
# Main79 core implementation
# =========================
import os, json, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
warnings.filterwarnings("ignore")

SEED = 42
MODEL_SEEDS = [42, 123, 777]
BATCH_SIZE = 32
MAX_LEN = 384
EPOCHS = 6
LR = 3e-4
WEIGHT_DECAY = 1e-3
GRAD_CLIP = 1.0
NUM_WORKERS = 0
EMB_DIM = 64
HIDDEN = 64
DROPOUT = 0.40
RESIDUAL_WEIGHT = 0.025
TRAIN_WEIGHT = 0.075
META_C = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_jsonl(path, labelled=True):
    rows=[]
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line: rows.append(json.loads(line))
    texts=[r["text"] for r in rows]
    ids=[r["id"] for r in rows]
    if labelled:
        labels=np.asarray([0 if r["label"]=="A" else 1 for r in rows], dtype=np.int64)
        return ids,texts,labels
    return ids,texts

def reconstruct_safe_teacher(texts, labels, oof_meta, val_meta):
    idx=np.arange(len(labels))
    train_idx,val_idx=train_test_split(
        idx,test_size=0.20,random_state=SEED,stratify=labels
    )
    assert oof_meta.shape==(len(train_idx),13), (oof_meta.shape,len(train_idx))
    assert val_meta.shape==(len(val_idx),13), (val_meta.shape,len(val_idx))

    meta_model=LogisticRegression(C=META_C,max_iter=5000,solver="lbfgs",random_state=SEED)
    meta_model.fit(oof_meta, labels[train_idx])

    teacher_train_raw=meta_model.decision_function(oof_meta).astype(np.float32)
    teacher_val_raw=meta_model.decision_function(val_meta).astype(np.float32)

    teacher_scale=np.percentile(np.abs(teacher_train_raw),95)
    if teacher_scale < 1e-6: teacher_scale=1.0

    teacher_all=np.empty(len(labels),dtype=np.float32)
    teacher_all[train_idx]=np.clip(teacher_train_raw/teacher_scale,-6,6)
    teacher_all[val_idx]=np.clip(teacher_val_raw/teacher_scale,-6,6)

    meta_mean=oof_meta.mean(0)
    meta_std=np.where(oof_meta.std(0)<1e-6,1.0,oof_meta.std(0))
    meta_all=np.empty((len(labels),13),dtype=np.float32)
    meta_all[train_idx]=np.clip((oof_meta-meta_mean)/meta_std,-6,6)
    meta_all[val_idx]=np.clip((val_meta-meta_mean)/meta_std,-6,6)

    val_acc=accuracy_score(labels[val_idx],teacher_val_raw>=0)
    print(f"Main79 teacher validation accuracy: {val_acc:.6f}")
    print(f"Teacher scale: {teacher_scale:.6f}")
    return train_idx,val_idx,meta_model,teacher_scale,meta_mean,meta_std,teacher_all,meta_all

class ResidualDataset(Dataset):
    def __init__(self, seqs, labels, meta_z):
        self.seqs=seqs; self.labels=np.asarray(labels,dtype=np.float32); self.meta_z=np.asarray(meta_z,dtype=np.float32)
    def __len__(self): return len(self.seqs)
    def __getitem__(self,i):
        x=np.asarray(self.seqs[i],dtype=np.int64)
        if len(x)>MAX_LEN: x=x[:MAX_LEN]
        n=len(x)
        ids=np.full(MAX_LEN,PAD_IDX,dtype=np.int64)
        mask=np.zeros(MAX_LEN,dtype=np.bool_)
        ids[:n]=x; mask[:n]=True
        ur=len(np.unique(x))/max(n,1)
        aux=np.concatenate([self.meta_z[i],np.asarray([ur,1-ur,np.log1p(n)/7.0],dtype=np.float32)]).astype(np.float32)
        return torch.from_numpy(ids),torch.from_numpy(mask),torch.tensor(self.labels[i]),torch.from_numpy(aux)

class ResidualBiGRU(nn.Module):
    def __init__(self,meta_dim):
        super().__init__()
        self.embedding=nn.Embedding(VOCAB_SIZE+1,EMB_DIM,padding_idx=PAD_IDX)
        self.gru=nn.GRU(EMB_DIM,HIDDEN,num_layers=1,batch_first=True,bidirectional=True)
        enc_dim=2*HIDDEN
        self.attn=nn.Sequential(nn.Linear(enc_dim,32),nn.Tanh(),nn.Linear(32,1))
        self.neural_proj=nn.Sequential(nn.Linear(3*enc_dim,64),nn.LayerNorm(64),nn.ReLU(),nn.Dropout(DROPOUT))
        self.meta_proj=nn.Sequential(nn.Linear(meta_dim+3,32),nn.LayerNorm(32),nn.ReLU(),nn.Dropout(DROPOUT))
        self.head=nn.Sequential(nn.Linear(96,48),nn.ReLU(),nn.Dropout(DROPOUT),nn.Linear(48,1))
    def forward(self,ids,mask,aux):
        e=self.embedding(ids); h,_=self.gru(e)
        mf=mask.unsqueeze(-1); denom=mf.sum(1).clamp_min(1)
        mean_pool=(h*mf).sum(1)/denom
        max_pool=h.masked_fill(~mf,-1e4).max(1).values
        scores=self.attn(h.float()).squeeze(-1).masked_fill(~mask,-1e4)
        w=torch.softmax(scores,dim=1)
        attn_pool=torch.sum(h.float()*w.unsqueeze(-1),dim=1)
        pooled=torch.cat([mean_pool.float(),max_pool.float(),attn_pool],1)
        z=torch.cat([self.neural_proj(pooled),self.meta_proj(aux.float())],1)
        return self.head(z).squeeze(-1)

def autocast_context():
    if USE_AMP:
        return torch.autocast(device_type="cuda",dtype=torch.float16)
    return torch.autocast(device_type="cpu",enabled=False)

def train_residual(model, loader, epochs=EPOCHS):
    model=model.to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    crit=nn.BCEWithLogitsLoss()
    scaler=torch.amp.GradScaler("cuda",enabled=USE_AMP)
    for ep in range(1,epochs+1):
        model.train(); total=0; correct=0; loss_sum=0
        t0=time.time()
        for ids,mask,y,aux in loader:
            ids=ids.to(DEVICE); mask=mask.to(DEVICE); y=y.to(DEVICE); aux=aux.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast_context():
                r=model(ids,mask,aux)
                loss=crit(aux[:,0]+TRAIN_WEIGHT*r,y)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            scaler.step(opt); scaler.update()
            fl=(aux[:,0]+TRAIN_WEIGHT*r).detach()
            correct += int(((fl>=0).long()==y.long()).sum())
            total += len(y); loss_sum += float(loss.item())*len(y)
        print(f"epoch {ep:02d}/{epochs} | acc={(correct/max(total,1)):.5f} | loss={(loss_sum/max(total,1)):.5f} | {(time.time()-t0)/60:.2f} min")
    return model

@torch.no_grad()
def predict_residual(model,seqs,meta_z):
    model.eval()
    ds=ResidualDataset(seqs,np.zeros(len(seqs),dtype=np.float32),meta_z)
    dl=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=USE_AMP)
    out=[]
    for ids,mask,_,aux in dl:
        ids=ids.to(DEVICE); mask=mask.to(DEVICE); aux=aux.to(DEVICE)
        with autocast_context():
            r=model(ids,mask,aux)
        out.append(r.float().cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def fit_residual_ensemble(train_seqs,train_y,train_meta_z,seeds=MODEL_SEEDS,epochs=EPOCHS):
    ds=ResidualDataset(train_seqs,train_y,train_meta_z)
    dl=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=USE_AMP)
    models=[]; preds=[]
    for seed in seeds:
        print("\n"+"="*70); print("Residual seed",seed); print("="*70)
        seed_everything(seed)
        model=train_residual(ResidualBiGRU(train_meta_z.shape[1]),dl,epochs)
        models.append(model)
    return models


## 4. Reconstruct the leakage-safe Main79 teacher

In [ ]:

seed_everything(42)
(train_idx,val_idx,teacher_model,teacher_scale,meta_mean,meta_std,
 teacher_all,meta_all)=reconstruct_safe_teacher(texts,labels,oof_meta,val_meta)

teacher_val_raw=teacher_model.decision_function(val_meta).astype(np.float32)
teacher_val_score=np.clip(teacher_val_raw/teacher_scale,-6,6).astype(np.float32)
print("Teacher val accuracy:",accuracy_score(labels[val_idx],teacher_val_score>=0))


## 5. Train Main79 residual ensemble on the 8,428 training rows and evaluate on 2,108 validation rows

In [ ]:

VOCAB_SIZE=max(int(t) for s in texts for t in s)+1
PAD_IDX=VOCAB_SIZE
print("Vocabulary size:",VOCAB_SIZE,"PAD:",PAD_IDX)

train_texts=[texts[i] for i in train_idx]
val_texts=[texts[i] for i in val_idx]
y_train=labels[train_idx]; y_val=labels[val_idx]
meta_train=meta_all[train_idx]; meta_val=meta_all[val_idx]

# For validation reproduction, train only on the 8428 training rows.
val_models=fit_residual_ensemble(train_texts,y_train,meta_train,epochs=EPOCHS)

val_residuals=[]
for seed,model in zip(MODEL_SEEDS,val_models):
    r=predict_residual(model,val_texts,meta_val)
    val_residuals.append(r)
val_residual_mean=np.mean(np.vstack(val_residuals),axis=0).astype(np.float32)

main79_val_score=teacher_val_score + RESIDUAL_WEIGHT*val_residual_mean
main79_val_pred=(main79_val_score>=0).astype(np.int64)

print("\nMAIN79 VALIDATION")
print("Teacher accuracy :",accuracy_score(y_val,teacher_val_score>=0))
print("Main79 accuracy  :",accuracy_score(y_val,main79_val_pred))
print("Main79 AUC       :",roc_auc_score(y_val,main79_val_score))
print("Residual mean/std:",float(val_residual_mean.mean()),float(val_residual_mean.std()))

np.save("/content/main79_val_teacher.npy",teacher_val_score)
np.save("/content/main79_val_residual.npy",val_residual_mean)
np.save("/content/main79_val_score.npy",main79_val_score)
np.save("/content/main79_val_predictions.npy",main79_val_pred)
np.save("/content/main79_val_labels.npy",y_val)


## 6. Build the exact Main66 full-data test teacher and Main79 test prediction

In [ ]:

# Exact Main66 full-data representation and base scores
(
    all_strings,all_transitions,full_tfidf,full_trans_vectorizer,
    full_compact_scaler,X_all_tfidf,X_all_compact,X_all_global
)=fit_global_representation(texts)

(
    test_strings,test_transitions,X_test_tfidf,X_test_compact,X_test_global
)=transform_global_representation(test_texts,full_tfidf,full_trans_vectorizer,full_compact_scaler)

print("Full global:",X_all_global.shape,"Test global:",X_test_global.shape)

test_svm=train_svm(X_all_global,labels,X_test_global)
test_nbsvm=train_nbsvm(all_strings,labels,test_strings)
test_hgb=train_hgb(X_all_compact,labels,X_test_compact)
test_local=local_geometry_features(X_all_tfidf,labels,X_test_tfidf,k=LOCAL_K)

# Reconstruct exact 13-D Main64 scaler from the saved OOF base scores.
oof_svm=np.load(PATHS["oof_svm"]).reshape(-1)
oof_nbsvm=np.load(PATHS["oof_nbsvm"]).reshape(-1)
oof_hgb=np.load(PATHS["oof_hgb"]).reshape(-1)
oof_local=np.load(PATHS["oof_local"])
if oof_local.ndim==1: oof_local=oof_local[:,None]

oof_local_b=-oof_local
oof_raw=np.column_stack([
    oof_svm,oof_nbsvm,oof_hgb,
    oof_local_b[:,0],oof_local_b[:,1],oof_local_b[:,2],oof_local_b[:,3],
    oof_local_b[:,6],oof_local_b[:,7],oof_local_b[:,8],oof_local_b[:,9],oof_local_b[:,10]
])
from sklearn.preprocessing import StandardScaler
exact_meta_scaler=StandardScaler().fit(oof_raw)
reconstructed=exact_meta_scaler.transform(oof_raw)
print("Meta reconstruction max diff:",
      float(np.max(np.abs(reconstructed-oof_meta[:,:12]))))
assert np.max(np.abs(reconstructed-oof_meta[:,:12]))<1e-5

test_meta=make_meta_features(test_svm,test_nbsvm,test_hgb,test_local,exact_meta_scaler).astype(np.float32)
test_teacher_raw=teacher_model.decision_function(test_meta).astype(np.float32)
test_teacher=np.clip(test_teacher_raw/teacher_scale,-6,6).astype(np.float32)
test_meta_z=np.clip((test_meta-meta_mean)/meta_std,-6,6).astype(np.float32)

# Main79 original full-data residual training convention: all 10536 rows, fixed 6 epochs.
full_models=fit_residual_ensemble(texts,labels,meta_all,epochs=EPOCHS)
test_res=[]
for seed,model in zip(MODEL_SEEDS,full_models):
    test_res.append(predict_residual(model,test_texts,test_meta_z))
test_residual_mean=np.mean(np.vstack(test_res),axis=0).astype(np.float32)
main79_test_score=test_teacher+RESIDUAL_WEIGHT*test_residual_mean
main79_test_pred=(main79_test_score>=0).astype(np.int64)

print("Test teacher range:",float(test_teacher.min()),float(test_teacher.max()))
print("Test residual std:",float(test_residual_mean.std()))
print("Pred A/B:",int((main79_test_pred==0).sum()),int((main79_test_pred==1).sum()))

submission_main79=pd.DataFrame({
    "id":test_ids,
    "label":np.where(main79_test_pred==0,"A","B")
})
submission_main79.to_csv("/content/submission_main79_reproduced.csv",index=False)

np.save("/content/main79_test_score.npy",main79_test_score)
np.save("/content/main79_test_teacher.npy",test_teacher)
np.save("/content/main79_test_residual.npy",test_residual_mean)
print("Saved /content/submission_main79_reproduced.csv")


## 7. Reproduction checklist

The important numbers to report back are:

- teacher validation accuracy;
- Main79 validation accuracy;
- Main79 validation AUC;
- prediction counts;
- meta reconstruction max difference.

Do **not** spend a Kaggle submission yet. The next notebook will use these diagnostics to build the Main79 + Exp11 stack.
